In [1]:
import pandas as pd

# ROOT
root_path = "C:/Users/jackm/Documents/GitHub/ms-capstone-TTS-natlang-styleprompts"

# ParaSpeechCaps

In [9]:
from datasets import load_dataset
import pandas as pd

# Load the entire dataset
# dataset = load_dataset("ajd12342/paraspeechcaps")

# Load specific splits of the dataset
# train_scaled = load_dataset("ajd12342/paraspeechcaps", split="train_scaled") #Only has EMilia files
train_base = load_dataset("ajd12342/paraspeechcaps", split="train_base")
holdout = load_dataset("ajd12342/paraspeechcaps", split="holdout")
dev = load_dataset("ajd12342/paraspeechcaps", split="dev")
test = load_dataset("ajd12342/paraspeechcaps", split="test")

dfs = [
    # train_scaled.to_pandas(),
    train_base.to_pandas(),
    holdout.to_pandas(),
    dev.to_pandas(),
    test.to_pandas(),
]

# merge into one dataframe
df = pd.concat(dfs, ignore_index=True)

c:\Users\jackm\miniconda3\envs\capstone-eda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
df.head()

,source,relative_audio_path,text_description,transcription,intrinsic_tags,situational_tags,basic_tags,all_tags,speakerid,name,duration,gender,accent,pitch,speaking_rate,noise,utterance_pitch_mean,snr,phonemes,tag_of_interest
0,voxceleb,voxceleb1/dev/wav/id11193/CbXwBsHMlbA/00001_vo...,[ A male speaker with an American accent deliv...,I did that from my heart. So that if they do ...,"[american, authoritative, flowing, raspy]",None,"[environment balanced in clarity, fast speed, ...","[american, authoritative, environment balanced...",id11193,Tupac Shakur,4.880063,male,american,medium-pitched,fast speed,environment balanced in clarity,146.627151,43.092300,aɪ dɪd ðæt fɹʌm maɪ hɑɹt. soʊ ðæt ɪf ðeɪ du t...,None
1,voxceleb,voxceleb2/dev/aac/id00708/s_qY2Q1gjaw/00415_vo...,"[ A deep-voiced man delivers authoritative, cr...","You know, people always refer to it as duct t...","[authoritative, american, crisp, deep, flowing]",None,"[environment balanced in clarity, low-pitched,...","[american, authoritative, crisp, deep, environ...",id00708,Antonin Scalia,7.800000,male,american,low-pitched,measured speed,environment balanced in clarity,102.925285,43.679882,"ju noʊ, pipʌl ɔlweɪz ɹʌfɜ˞ tu ɪt æz dʌkt teɪp...",None
2,voxceleb,voxceleb2/dev/aac/id05211/Qpv7DIHWDBA/00073_vo...,[ A female speaker with a medium-pitched voice...,lisa plays this like live floor tom um which ...,[australian],None,"[female, medium-pitched, noisy environment, sl...","[australian, female, medium-pitched, noisy env...",id05211,Lisa Origliasso,7.700000,female,australian,medium-pitched,slow speed,noisy environment,171.683182,28.372160,lisʌ pleɪz ðɪs laɪk laɪv flɔɹ tɑm ʌm wɪtʃ dʒʌ...,None
3,voxceleb,voxceleb2/test/aac/id05055/cZZMU1w_HO4/00349_v...,"[ A male speaker with a deep, low-pitched Amer...",And my coaches and teammates are going to hel...,"[deep, american]",None,"[fast speed, low-pitched, male, very noisy env...","[american, deep, fast speed, low-pitched, male...",id05055,LeBron James,7.900000,male,american,low-pitched,fast speed,very noisy environment,144.090042,21.178427,ʌnd maɪ koʊtʃɪz ʌnd timmeɪts ɑɹ ɡoʊɪŋ tu hɛlp...,None
4,voxceleb,voxceleb2/dev/aac/id03757/BP4bXjDuMFE/00044_vo...,[ A male speaker's voice is nasal and sing-son...,other people had had around me growing up in ...,"[nasal, singsong, american]",None,"[male, measured speed, medium-pitched, very no...","[american, male, measured speed, medium-pitche...",id03757,James Franco,4.700000,male,american,medium-pitched,measured speed,very noisy environment,109.919975,3.625704,ʌðɜ˞ pipʌl hæd hæd ɜ˞aʊnd mi ɡɹoʊɪŋ ʌp ɪn pæl...,None


In [10]:
# Duration in hours
total_hours = df["duration"].sum() / 3600
print(total_hours)

23.082813877314813


In [8]:
dur_by_source = df.groupby('source')['duration'].sum().reset_index()
dur_by_source["duration_hrs"] = dur_by_source['duration'] / 3600

dur_by_source

,source,duration,duration_hrs
0,ears,218409.000000,60.669167
1,expresso,109197.553312,30.332654
2,voxceleb,904715.809875,251.309947


In [11]:
df[df["source"]== "voxceleb"]

pd.set_option('display.max_colwidth', None)
df["text_description"]

0         [ A male speaker with an American accent delivers authoritative and flowing speech at a fast speed. His voice is raspy and medium-pitched, recorded in an environment that provides good balance in clarity.]
1                          [ A deep-voiced man delivers authoritative, crisp speech with a low pitch, recorded in an environment balanced in clarity. His measured speed and flowing rhythm convey an American accent.]
2                                                                          [ A female speaker with a medium-pitched voice delivers her speech in a slow manner against the backdrop of a noisy Australian environment.]
3                                                                                            [ A male speaker with a deep, low-pitched American accent delivers his speech at a fast pace in a very noisy environment.]
4                                                                            [ A male speaker's voice is nasal and sing-song in tone, sp

In [ ]:
# visualize tag distribution

import pandas as pd
import numpy as np

def is_populated(x):
    if x is None:
        return False
    if isinstance(x, float) and np.isnan(x):
        return False
    if isinstance(x, str):
        return x != 'None' and x.strip() != ''
    if isinstance(x, (list, tuple, np.ndarray)):
        return len(x) > 0
    return True  # fallback for other types

df['has_intrinsic'] = df['intrinsic_tags'].apply(is_populated)
df['has_situational'] = df['situational_tags'].apply(is_populated)

def categorize(row):
    if row['has_intrinsic'] and row['has_situational']:
        return 'both'
    elif row['has_intrinsic']:
        return 'intrinsic_only'
    elif row['has_situational']:
        return 'situational_only'
    else:
        return 'neither'

df['tag_category'] = df.apply(categorize, axis=1)

counts = (
    df.groupby(['source', 'tag_category'])
      .size()
      .unstack(fill_value=0)
)

import matplotlib.pyplot as plt

counts.plot(kind='bar')

plt.title('Tag Presence by Source')
plt.xlabel('Source')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Category')
plt.tight_layout()

plt.show()

# StyleTalk

In [6]:
styletalk_root = f"../data/raw/styletalk"

In [8]:
import pandas as pd
df_styletalk_eval_meta = pd.read_csv(
    f"{styletalk_root}/annotations/eval.csv",
    sep=",",            # delimiter
    encoding="utf-8",   # or "latin1"
    na_values=["NA", ""],
)

df_styletalk_eval_meta

,diag_id,context,curr_audio_id,res_audio_id,curr_text,res_text,curr_emotion,curr_speed,curr_volume,res_emotion,res_speed,res_volume
0,health_306,"A : I've been trying to eat healthier lately, ...",health_306/c_0.wav,health_306/r_0.wav,B: Sometimes it's hard to stick to those habit...,"Yeah, it can be a real challenge when life g...",unfriendly,fast,loud,unfriendly,normal,normal
1,health_306,"A : I've been trying to eat healthier lately, ...",health_306/c_2.wav,health_306/r_2.wav,B: Sometimes it's hard to stick to those habit...,"True, I'm hoping I can make these changes a ...",friendly,normal,normal,cheerful,normal,normal
2,music_478,"A : Oh, have you heard the new album by The We...",music_478/c_0.wav,music_478/r_0.wav,"B: Hmm, I might go to their concert next month.","Oh, that could be fun. If you're going, mayb...",neutral,normal,normal,neutral,normal,normal
3,movie_430,"A : Seriously, that last movie we saw was just...",movie_430/c_1.wav,movie_430/r_1.wav,B: I just hope the next one we pick doesn't en...,"Whoa, take it easy! We'll make sure it's a h...",unfriendly,fast,loud,unfriendly,normal,normal
4,movie_48,A : Did you see the latest superhero movie? It...,movie_48/c_2.wav,movie_48/r_2.wav,"B: I just prefer movies with more depth, you k...","Absolutely, there's plenty of films with dep...",friendly,normal,loud,cheerful,normal,normal
...,...,...,...,...,...,...,...,...,...,...,...,...
853,game_408,"A : I can't believe we aced that level, that w...",game_408/c_0.wav,game_408/r_0.wav,"B: Sure, just give me a second to set things u...","No worries, take your time, we're not in a r...",neutral,slow,quiet,friendly,normal,normal
854,food_233,A : I can't believe how amazing that new Thai ...,food_233/c_0.wav,food_233/r_0.wav,B: They did seem to have a lot of options on t...,"Exactly! There's so much to try, maybe anoth...",neutral,normal,normal,cheerful,normal,normal
855,music_21,"A : I just found this amazing new band, and th...",music_21/c_2.wav,music_21/r_2.wav,B: I think I've heard about them once before.,That's cool! They've been gaining popularity...,friendly,normal,normal,cheerful,normal,normal
856,music_21,"A : I just found this amazing new band, and th...",music_21/c_1.wav,music_21/r_1.wav,B: I think I've heard about them once before.,"Wait, why do you sound mad? Did you not like...",unfriendly,fast,loud,unfriendly,normal,loud


In [ ]:
# Find total duration of StyleTalk: RESULTS = 7 hours

"""
Recursively sum the total duration of all WAV files in a directory.
Usage: python sum_wav_durations.py [directory]
"""

import sys
import wave
import os
from pathlib import Path
from tqdm import tqdm

# Sonnet 4.6: 

def get_wav_duration(filepath: Path) -> float:
    """Return duration of a WAV file in seconds, or None on error."""
    try:
        with wave.open(str(filepath), "rb") as f:
            frames = f.getnframes()
            rate = f.getframerate()
            return frames / rate
    except Exception as e:
        tqdm.write(f"[WARN] Could not read {filepath}: {e}")
        return None


def format_duration(seconds: float) -> str:
    """Format seconds into a human-readable string."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    parts = []
    if h:
        parts.append(f"{h}h")
    if m or h:
        parts.append(f"{m}m")
    parts.append(f"{s:.2f}s")
    return " ".join(parts)


def scan_directory(root: Path):
    total_seconds = 0.0
    file_count = 0
    error_count = 0

    wav_files = sorted(root.rglob("*.wav")) + sorted(root.rglob("*.WAV"))
    # Deduplicate (case-sensitive filesystems may return both)
    seen = set()
    unique_wavs = []
    for f in wav_files:
        if f not in seen:
            seen.add(f)
            unique_wavs.append(f)

    if not unique_wavs:
        print(f"No WAV files found under: {root}")
        return

    print(f"Scanning: {root}\n")

    with tqdm(unique_wavs, unit="file", dynamic_ncols=True) as pbar:
        for filepath in pbar:
            rel = filepath.relative_to(root)
            pbar.set_postfix_str(str(rel), refresh=True)

            duration = get_wav_duration(filepath)
            if duration is not None:
                total_seconds += duration
                file_count += 1
            else:
                error_count += 1

    print(f"\n{'─' * 60}")
    print(f"  Files found   : {file_count + error_count}")
    print(f"  Files read    : {file_count}")
    if error_count:
        print(f"  Errors        : {error_count}")
    print(f"  Total duration: {format_duration(total_seconds)}  ({total_seconds:.2f}s)")



directory = Path(f"{styletalk_root}/audio")

if not directory.exists():
    print(f"Error: directory not found: {directory}")
    sys.exit(1)
if not directory.is_dir():
    print(f"Error: not a directory: {directory}")
    sys.exit(1)

scan_directory(directory)

Scanning: C:\Users\jackm\Documents\GitHub\ms-capstone-TTS-natlang-styleprompts\src\data\raw\styletalk\audio



100%|██████████| 5392/5392 [00:35<00:00, 153.77file/s, work_98\r_2.wav]         



────────────────────────────────────────────────────────────
  Files found   : 5392
  Files read    : 5392
  Total duration: 7h 7m 54.19s  (25674.19s)
